# Семинар 07. Протоколы и duck typing


## Цели

После семинара вы сможете:

- различать номинальную и структурную типизацию;
- описывать интерфейсы с ABC и typing.Protocol;
- понимать, что аннотации и протоколы проверяют до запуска и во время выполнения.

## Перед началом

Повторите наследование, абстрактные классы и аннотации типов. Для демонстрации типов нужен nb-mypy.


## Полезные ссылки

- [Python: `typing.Protocol` и `runtime_checkable`](https://docs.python.org/3/library/typing.html#typing.Protocol)
- [Спецификация типизации: протоколы](https://typing.python.org/en/latest/spec/protocol.html)
- [mypy: protocols and structural subtyping](https://mypy.readthedocs.io/en/stable/protocols.html)

## Динамическая и статическая проверка типов

В Python тип принадлежит объекту, а одно имя в разные моменты может ссылаться на объекты разных типов. Аннотация описывает, какой тип ожидается в этом месте, но интерпретатор обычно не заставляет программу соблюдать это описание.

Аннотация похожа на маркировку коробки: надпись сообщает, что должно лежать внутри, но физически не мешает положить другой предмет. Статический анализатор сверяет такие маркировки до запуска программы и находит часть несовместимостей заранее. Во время выполнения работают уже реальные объекты и операции над ними.

```python
class Rectangle:
    def __init__(self, width: int, height: int) -> None:
        self._width = width
        self._height = height

    def get_area(self) -> int:
        return self._width * self._height


rectangle = Rectangle(10, "3")  # ошибка статической типизации
print(rectangle.get_area())
```

Этот код запускается: Python умеет умножать строку на целое число, поэтому вместо площади программа печатает строку `3333333333`. Аннотация не изменила поведение во время выполнения. Статический анализатор, напротив, увидит строку там, где конструктор ожидает `int`, и сообщит об ошибке до запуска.

## Номинальная и структурная совместимость

| Подход | Когда объект считается совместимым | Инструмент Python |
|---|---|---|
| Номинальный | его класс явно входит в нужную иерархию наследования | обычный базовый класс, `abc.ABC`, `isinstance()` |
| Структурный | объект предоставляет требуемые методы и атрибуты совместимых типов | `typing.Protocol` и статический анализатор |

При номинальной проверке важна официальная принадлежность класса к иерархии: анализатор смотрит, от какого типа класс унаследован. При структурной проверке происхождение не важно — проверяется набор доступных операций и их типы.

Обычный duck typing проявляется во время выполнения: код вызывает нужный метод и не требует заранее предъявить принадлежность к определённому классу. `Protocol` описывает тот же требуемый набор операций для статического анализатора.

Проверка `isinstance()` отвечает на ограниченный вопрос о типе или иерархии объекта во время выполнения. Она не проверяет аннотации всех методов и не заменяет ни валидацию входных данных, ни статический анализ.

## Абстрактные базовые классы

`abc.ABC` задаёт номинальный контракт для явных наследников. Конкретный наследник должен реализовать все абстрактные методы, иначе Python не позволит создать его экземпляр. ABC подходит, когда классы образуют одну управляемую иерархию, а вместе с интерфейсом нужно передать общую реализацию или состояние.


In [ ]:
from abc import ABC, abstractmethod


class Flyable(ABC):
    @abstractmethod
    def fly(self) -> None:
        ...


class Junkie(ABC):
    @abstractmethod
    def fly(self) -> None:
        ...


class Bird(Flyable):
    def fly(self) -> None:
        print("I'm flying")


class Hippie(Junkie):
    def fly(self) -> None:
        print("I'm flying high")


class FlyableManager:
    def __init__(self) -> None:
        self.flyables: list[Flyable] = []

    def add_flyable(self, flyable: Flyable) -> None:
        self.flyables.append(flyable)
    
    def fly(self) -> None:
        for flyable in self.flyables:
            flyable.fly()


manager = FlyableManager()
manager.add_flyable(Bird())
# Статический анализатор отклонит строку ниже: Hippie не наследует Flyable.
# Во время выполнения явной проверки нет, поэтому вызов сработает.
manager.add_flyable(Hippie())
manager.fly()

В примере `Hippie` реализует метод `fly()`, но через `Junkie` принадлежит другой номинальной иерархии. Поэтому статический анализатор отклоняет передачу `Hippie` в `add_flyable()`. Во время выполнения вызов всё же проходит: аннотация параметра не запускает автоматическую проверку типа. Когда менеджер вызывает `fly()`, нужный метод у объекта есть.

## Протоколы и duck typing

`Protocol` перечисляет методы и атрибуты, которые нужны клиентскому коду. Явное наследование не требуется: анализатор считает класс совместимым, если все члены протокола присутствуют и имеют совместимые типы. Класс не обязан заранее знать о протоколе, под который подходит. Так статическая проверка формализует duck typing, не связывая реализации общей иерархией.


In [ ]:
%load_ext nb_mypy
from typing import Protocol


class Flyable(Protocol):
    def fly(self) -> None:
        ...


class Bird:
    def fly(self) -> None:
        print("I'm flying")


class Hippie:
    def fly(self) -> None:
        print("I'm flying high")


class ParametrizedFlyer:
    def fly(self, speed: int) -> None:
        print(f"I'm flying at {speed} knots")


class FlyableManager:
    def __init__(self) -> None:
        self.flyables: list[Flyable] = []

    def add_flyable(self, flyable: Flyable) -> None:
        self.flyables.append(flyable)
    
    def fly(self) -> None:
        for flyable in self.flyables:
            flyable.fly()


manager = FlyableManager()
manager.add_flyable(Bird())
# Hippie не наследуется от Flyable, но структурно соответствует протоколу.
manager.add_flyable(Hippie())
manager.fly()

# Сигнатура fly несовместима: обязательный аргумент speed отсутствует в протоколе.
# manager.add_flyable(ParametrizedFlyer())
# У int вообще нет метода fly.
# manager.add_flyable(1)


## Проверка протокола во время выполнения

Обычный протокол нельзя передать вторым аргументом в `isinstance()`. Декоратор `@runtime_checkable` разрешает такую проверку, но во время выполнения Python проверяет только наличие членов протокола. Он видит метод с нужным именем, но не сверяет типы и количество его параметров или тип результата. Поэтому объект с несовместимой сигнатурой может пройти `isinstance()`, а ошибка проявится только при вызове метода.


In [ ]:
from typing import Protocol, runtime_checkable


@runtime_checkable
class RuntimeFlyable(Protocol):
    def fly(self) -> None:
        ...


print(isinstance(Bird(), RuntimeFlyable))
# Метод fly существует, но проверка во время выполнения не видит его обязательный параметр.
print(isinstance(ParametrizedFlyer(), RuntimeFlyable))


## Как выбрать инструмент

| Ситуация | Подход |
|---|---|
| Классы образуют одну управляемую иерархию, нужна общая реализация | базовый класс или `ABC` |
| Функции достаточно небольшого набора операций от любых подходящих объектов | узкий `Protocol` |
| Код прост и полностью покрыт тестами, статическая проверка не используется | обычный duck typing |
| Нужно проверить внешние данные во время выполнения | явная валидация данных; одного `Protocol` недостаточно |

Интерфейс должен описывать ровно те возможности, которыми пользуется клиент. Если функции нужен только `fly()`, протокол из одного метода честнее типа с десятками несвязанных операций. Узкий интерфейс проще реализовать, проверить и заменить в тесте.


## Самопроверка

1. Почему аннотация `height: int` не останавливает передачу строки во время выполнения?
2. Почему `Hippie` несовместим с ABC `Flyable`, хотя у него есть метод `fly()`?
3. Почему тот же `Hippie` совместим с протоколом `Flyable`?
4. Как анализатор обнаруживает несовместимость `ParametrizedFlyer` с протоколом?
5. Почему `@runtime_checkable` нельзя использовать для полной проверки сигнатуры?


## Итоги

- Аннотации помогают анализатору, но обычно не проверяются интерпретатором автоматически.
- ABC задаёт номинальную совместимость через наследование.
- `Protocol` задаёт структурную совместимость через набор членов и их типы.
- Duck typing работает во время выполнения независимо от наличия `Protocol`.
- `@runtime_checkable` проверяет наличие членов, но игнорирует их типовые сигнатуры.


## Задание 1. Круглые скобки (1 балл)

Строка состоит только из `(` и `)`. Определите, является ли она правильной скобочной последовательностью.

```python
def is_valid_parentheses(text: str) -> bool:
    ...
```

**Примеры:** `""`, `"()"` и `"(())()"` валидны; `")("` и `"(()"` невалидны.

**Критерии проверки:** `O(N)` по времени, `O(1)` по дополнительной памяти; промежуточный баланс никогда не становится отрицательным и в конце равен нулю.


## Задание 2. Несколько видов скобок (2 балла)

Строка состоит из символов `()[]{}`. Закрывающая скобка должна соответствовать последней незакрытой скобке.

```python
def is_valid_brackets(text: str) -> bool:
    ...
```

**Примеры:** `"[]()"` и `"{[()]}"` валидны; `"[(])"` и `"{{}"` невалидны.

**Критерии проверки:** `O(N)` по времени и `O(N)` по памяти; решение корректно обрабатывает пустую строку и закрывающую скобку в начале.
